# Google Play Developer Monetization and Distribution Performance

```SQL
DROP TABLE IF EXISTS fct_app_revenue;
DROP TABLE IF EXISTS dim_app_category;
DROP TABLE IF EXISTS  dim_monetization_model;

CREATE TABLE fct_app_revenue (
    app_id integer,
    category_id integer,
    revenue_date date,
    revenue_amount decimal
);

CREATE TABLE dim_app_category (
    category_id integer,
    category_name varchar
);

CREATE TABLE dim_monetization_model (
    app_id integer,
    monetization_type varchar
);

INSERT INTO fct_app_revenue (app_id, category_id, revenue_date, revenue_amount)
VALUES
    (101, 1, '2024-04-05', 100),
    (101, 1, '2024-04-12', 150),
    (101, 1, '2024-05-10', 130),
    (101, 1, '2024-06-05', 120),
    (102, 2, '2024-04-07', 80),
    (102, 2, '2024-05-14', 90),
    (102, 2, '2024-06-12', 85),
    (103, 3, '2024-04-15', 200),
    (103, 3, '2024-05-19', 220),
    (104, 4, '2024-04-10', 50),
    (104, 4, '2024-04-25', 70),
    (104, 4, '2024-05-23', 60),
    (104, 4, '2024-06-15', 80),
    (105, 5, '2024-04-02', 90),
    (105, 5, '2024-05-04', 100),
    (105, 5, '2024-06-28', 95),
    (106, 6, '2024-04-18', 110),
    (106, 6, '2024-05-11', 115),
    (106, 6, '2024-06-22', 120),
    (107, 7, '2024-04-22', 140),
    (107, 7, '2024-05-15', 130),
    (107, 7, '2024-06-17', 150),
    (108, 8, '2024-04-28', 60),
    (108, 8, '2024-05-21', 55),
    (109, 9, '2024-04-13', 75),
    (109, 9, '2024-05-27', 65),
    (109, 9, '2024-06-10', 80),
    (110, 10, '2024-04-08', 95),
    (110, 10, '2024-05-30', 105),
    (110, 10, '2024-06-25', 115);

INSERT INTO dim_app_category (category_id, category_name)
VALUES
    (1, 'Games'),
    (2, 'Productivity'),
    (3, 'Social'),
    (4, 'Entertainment'),
    (5, 'Education'),
    (6, 'Health & Fitness'),
    (7, 'Finance'),
    (8, 'Photography'),
    (9, 'Utilities'),
    (10, 'Travel');

INSERT INTO dim_monetization_model (app_id, monetization_type)
VALUES
    (101, 'subscription'),
    (102, 'ads'),
    (103, 'in-app purchase'),
    (104, 'subscription'),
    (105, 'ads'),
    (106, 'in-app purchase'),
    (107, 'subscription'),
    (108, 'ads'),
    (109, 'subscription'),
    (110, 'in-app purchase');

SELECT * FROM fct_app_revenue;
SELECT * FROM dim_app_category;
SELECT * FROM dim_monetization_model;
```

In [1]:
import pandas as pd
import numpy as np

In [2]:
df_category = pd.read_csv('Data/006/dim_app_category.csv')
df_monetizacion = pd.read_csv('Data/006/dim_monetization_model.csv')
df_revenue = pd.read_csv('Data/006/fct_app_revenue.csv', parse_dates=['revenue_date'])

df_category.head()

,category_id,category_name
0,1,Games
1,2,Productivity
2,3,Social
3,4,Entertainment
4,5,Education


In [3]:
df_monetizacion.head()

,app_id,monetization_type
0,101,subscription
1,102,ads
2,103,in-app purchase
3,104,subscription
4,105,ads


In [4]:
df_revenue.head()

,app_id,category_id,revenue_date,revenue_amount
0,101,1,2024-04-05,100
1,101,1,2024-04-12,150
2,101,1,2024-05-10,130
3,101,1,2024-06-05,120
4,102,2,2024-04-07,80


# Pregunta 1

Para el mes de abril de 2024, ¿cuáles fueron las categorías de aplicaciones que generaron los ingresos totales más altos (solo las 10 mejores)? Esta información se utilizará para perfeccionar las estrategias de monetización para los desarrolladores.

In [6]:
# 1. Unir los dataframes
df_full = df_revenue.merge(df_category, on='category_id')

# 2. Filtrar abril
df_abril = df_full[df_full['revenue_date'].between('2024-04-01', '2024-04-30')]

# 3. Agrupar, sumar y obtener el Top 10
top_10_categorias = (
    df_abril.groupby('category_name')['revenue_amount']
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)

top_10_categorias

,category_name,revenue_amount
0,Games,250
1,Social,200
2,Finance,140
3,Entertainment,120
4,Health & Fitness,110
5,Travel,95
6,Education,90
7,Productivity,80
8,Utilities,75
9,Photography,60


```SQL
SELECT 
    c.category_name, 
    SUM(r.revenue_amount) AS total_revenue
FROM fct_app_revenue r
JOIN dim_app_category c ON r.category_id = c.category_id
WHERE r.revenue_date BETWEEN '2024-04-01' AND '2024-04-30'
GROUP BY c.category_name
ORDER BY total_revenue DESC
LIMIT 10;
```